# Stuff to remember
1. To run this start jupyter lab or whatever from the correct conda environment gee_xarray_env
2. if ee.Authenticate() gives issue, use `earthengine authenticate` command on the command line, after loading the correct conda env, to preauthenticate  

In [ ]:
import ee
import xarray as xr
import xee                # <-- Import xee explicitly
import rioxarray          # Adds rio accessor to xarray
import pandas as pd
import os
import time
import dask # Dask is used by xee but often doesn't need direct interaction here
import gc
import json
from dask.diagnostics import ProgressBar

In [ ]:
# --- Configuration ---
START_DATE = '2019-01-01'
END_DATE = '2024-12-31' # Inclusive end date for month calculation
AOI_NAME = 'Arizona'
OUTPUT_DIR = './output' # Directory to save the output files
COLLECTION_ID = 'COPERNICUS/S2_SR_HARMONIZED'
BANDS = ['S4', 'S8', 'QA60'] # Red, NIR, Quality Assessment
SCALE = 30 # Resolution in meters (native sentinal resoultion)
MAX_CLOUD_COVER = 40 # Max % cloud cover for image metadata filter
CRS = 'EPSG:3857' # WGS 84 coordinate system... for xarray scale to be in m

gcloud_project_id = '' # <-- Set your GCP project ID here

In [ ]:
# --- Earth Engine Authentication and Initialization ---
try:
    # Specify your Google Cloud Project ID here.
    # Find this in your Google Cloud Console (console.cloud.google.com).

    ee.Authenticate()  
    if gcloud_project_id == 'YOUR_GOOGLE_CLOUD_PROJECT_ID':
         print("WARNING: Please replace 'YOUR_GOOGLE_CLOUD_PROJECT_ID' with your actual Google Cloud Project ID in the script.")
         # Optionally, you could exit or raise an error here if the placeholder wasn't changed.
         # exit("Exiting: Google Cloud Project ID not set.")

    # Initialize Earth Engine, explicitly providing the Cloud Project ID.
    # The default 'opt_url' (standard API endpoint) is usually best for this script.
    # ee.Initialize(project=gcloud_project_id)

    # Example if you specifically needed the high-volume endpoint (usually not needed here):
    ee.Initialize(
        project=gcloud_project_id,
        opt_url='https://earthengine-highvolume.googleapis.com'
    )

    print(f"Earth Engine Initialized Successfully using project: {gcloud_project_id}")

except Exception as e:
    # Catch potential errors during initialization more specifically if needed
    if 'cloud project not found' in str(e).lower() or 'user does not have permission' in str(e).lower():
         print(f"Earth Engine initialization failed: Could not use project '{gcloud_project_id}'. "
               f"Ensure the ID is correct and your account has permissions. Error: {e}")
    else:
        print(f"Earth Engine initialization failed: {e}")

    print("Please also ensure you have authenticated via 'earthengine authenticate'")
    exit()



In [ ]:
# Scale Sentinel-2 Surface Reflectance bands
def scale_bands(image):
    opticalBands = image.select(['B4', 'B8']).multiply(0.0001)
    return image.addBands(opticalBands, None, True)

# Cloud mask for Sentinel-2 Surface Reflectance
def mask_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10  # opaque clouds
    cirrus_bit_mask = 1 << 11 # cirrus clouds
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask)
# # SCL values of interest:

# 3: Cloud shadows

# 8: Medium probability clouds

# 9: High probability clouds

# 10: Thin cirrus

# 11: Snow or ice
def mask_scl(image):
    scl = image.select('SCL')
    # Mask out clouds, shadows, snow, and water
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return image.updateMask(mask)

# Add NDVI band (B8 is NIR, B4 is red)
def add_ndvi(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)

In [ ]:
# --- Define Area of Interest (AOI) ---
try:
    states = ee.FeatureCollection('TIGER/2018/States')
    aoi = states.filter(ee.Filter.eq('NAME', AOI_NAME)).geometry()
    print(f"AOI for {AOI_NAME} obtained.")
except Exception as e:
    print(f"Error obtaining AOI for {AOI_NAME}: {e}")
    exit()

# --- Create Output Directory and Log File ---
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Monthly output files will be saved to: '{OUTPUT_DIR}'")

LOG_FILE = os.path.join(OUTPUT_DIR, "download_log.jsonl")
SUMMARY_FILE = os.path.join(OUTPUT_DIR, "download_summary.txt")

# --- Load existing log into a dict of successful filenames ---
file_log = {}
if os.path.exists(LOG_FILE):
    with open(LOG_FILE, 'r') as f:
        for line in f:
            try:
                record = json.loads(line.strip())
                if record.get("status") == "success":
                    file_log[record["filename"]] = "success"
            except json.JSONDecodeError:
                continue

# --- Define log append function ---
def append_to_log(filename, status):
    log_entry = {
        "filename": filename,
        "status": status,
        "timestamp": time.strftime('%Y-%m-%dT%H:%M:%S')
    }
    with open(LOG_FILE, 'a') as f:
        f.write(json.dumps(log_entry) + '\n')
        f.flush()
        os.fsync(f.fileno())

# --- Main Processing Loop ---
print("\nStarting yearly processing and monthly download/saving...")
start_year = int(START_DATE.split('-')[0])
end_year = int(END_DATE.split('-')[0])
processed_files_count = 0
skipped_files = 0
failed_files = 0
failed_filenames = []
total_start_time = time.time()

for year in range(start_year, end_year + 1):
    year_start_time = time.time()
    print(f"\n===== Processing Year: {year} =====")
    monthly_images_for_year = []

    for month in range(1, 12 + 1):
        current_start_date = f'{year}-{month:02d}-01'
        current_end_date = pd.to_datetime(current_start_date) + pd.offsets.MonthEnd(0)
        if pd.to_datetime(current_start_date) > pd.to_datetime(END_DATE):
            continue

        filter_start = max(pd.to_datetime(START_DATE), pd.to_datetime(current_start_date)).strftime('%Y-%m-%d')
        filter_end = min(pd.to_datetime(END_DATE) + pd.Timedelta(days=1),
                         current_end_date + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        if filter_start >= filter_end:
            continue

        print(f"  Defining EE median for: {year}-{month:02d}")
        try:
            # Sentinel-2 SR Harmonized
            s2_monthly = ee.ImageCollection(COLLECTION_ID).filterBounds(aoi).filterDate(filter_start, filter_end).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_COVER)).map(add_ndvi)  # Must be your NDVI function for B8, B4
            
            median_image = s2_monthly.select('NDVI').median().toFloat().clip(aoi)
            month_start_millis = ee.Date(current_start_date).millis()
            median_image = median_image.set('system:time_start', month_start_millis).set('year', year).set('month', month)
            
            # # --- DEBUG: Pixel count check ---
            # print("    -> Performing quick pixel count check...")
            # pixel_count = median_image.reduceRegion(
            #     reducer=ee.Reducer.count(), geometry=aoi, scale=SCALE * 100,
            #     maxPixels=1e8, bestEffort=True
            # ).get('NDVI')
            # count_value = pixel_count.getInfo()
            # print(f"    -> Pixel count check result: {count_value}")

            # if count_value is None or count_value == 0:
            #     print(f"    -> SKIPPING {year}-{month:02d} based on pixel count.")
            #     continue

            # # --- DEBUG: Optional EE Min/Max check ---
            # try:
            #     print("    -> Performing EE Min/Max check...")
            #     min_max_stats = median_image.reduceRegion(
            #         reducer=ee.Reducer.minMax(), geometry=aoi, scale=SCALE * 10,
            #         maxPixels=1e13, bestEffort=True
            #     ).getInfo()
            #     print(f"    -> EE Min/Max check result: {min_max_stats}")
            # except Exception as stats_e:
            #     print(f"    -> Warning: EE Min/Max check failed: {stats_e}")
            
            monthly_images_for_year.append(median_image)

        except Exception as e:
            print(f"    -> ERROR defining {year}-{month:02d}: {e}")

    if not monthly_images_for_year:
        print(f"No valid monthly images found for year {year}. Skipping download.")
        continue

    year_collection = ee.ImageCollection.fromImages(monthly_images_for_year)
    num_images_in_year = year_collection.size().getInfo()
    
    
    print(f"\nCreated ee.ImageCollection for {year} with {num_images_in_year} images.")

    # --- Download and Save Monthly Files ---
    try:
        print(f"Opening yearly collection ({year}) with xarray...")
        ds_year = xr.open_dataset(
            year_collection, engine='ee', geometry=aoi, scale=SCALE, crs=CRS)
        ds_year = ds_year.chunk({'time': 1})
        print(f"Dataset for {year} opened.")

        # Ensure 'time' coordinate is correct type
        if 'time' in ds_year.coords and ds_year['time'].dtype != 'datetime64[ns]':
            ds_year['time'] = ds_year['time'].astype('datetime64[ns]')
        elif 'time' not in ds_year.coords:
            print("   Error: 'time' coordinate missing. Cannot save monthly files.")
            continue

        for time_step in ds_year['time']:
            month_save_start_time = time.time()
            dt = pd.to_datetime(time_step.values)
            save_year, save_month = dt.year, dt.month
            filename = f"{AOI_NAME}_NDVI_median_Sentinel2_{save_year}_{save_month:02d}.tif"
            filepath = os.path.join(OUTPUT_DIR, filename)

            # Skip if already logged as saved
            if file_log.get(filename) == "success":
                print(f"  Skipping already saved file: {filename}")
                skipped_files += 1
                continue

            print(f"  Saving: {filename}...")
            try:
                monthly_data_slice = ds_year['NDVI'].sel(time=time_step)

                # Ensure CRS is written
                if monthly_data_slice.rio.crs is None:
                    monthly_data_slice = monthly_data_slice.rio.write_crs(CRS)
                
                # Rename dims for consistency with rasterio
                if 'lat' in monthly_data_slice.dims:
                    monthly_data_slice = monthly_data_slice.rename({'lat': 'y', 'lon': 'x'})
                if 'X' in monthly_data_slice.dims:
                    monthly_data_slice = monthly_data_slice.rename({'X': 'x', 'Y': 'y'})
                
                # Make sure the order is (y, x)
                if 'y' in monthly_data_slice.dims and 'x' in monthly_data_slice.dims:
                    expected_order = ('y', 'x')
                    current_spatial_order = tuple(d for d in monthly_data_slice.dims if d in expected_order)
                    if current_spatial_order != expected_order:
                        print(f"    -> Transposing dimensions...")
                        full_transpose_order = list(expected_order) + [
                            d for d in monthly_data_slice.dims if d not in expected_order
                        ]
                        monthly_data_slice = monthly_data_slice.transpose(*full_transpose_order)

                # Optional debug
                print("Before squeeze:", monthly_data_slice.dims)
                monthly_data_slice = monthly_data_slice.squeeze()
                print("After squeeze:", monthly_data_slice.dims)
                
                with ProgressBar():
                    # Debug: load slice into memory
                    print("    -> Computing data slice stats (requires loading slice to RAM)...")
                    slice_data = monthly_data_slice.load()
                    min_val = slice_data.min().item()
                    max_val = slice_data.max().item()
                    mean_val = slice_data.mean().item()
                    nan_count = slice_data.isnull().sum().item()
                    total_pixels = slice_data.size
                    print(f"    -> Data slice stats: Min={min_val:.4f}, "
                          f"Max={max_val:.4f}, Mean={mean_val:.4f}, "
                          f"NaN_count={nan_count}/{total_pixels}")

                    # Remove scale/offset if present
                    slice_data.attrs.pop('scale_factor', None)
                    slice_data.attrs.pop('add_offset', None)
                    if hasattr(slice_data, 'encoding'):
                        slice_data.encoding.pop('scale_factor', None)
                        slice_data.encoding.pop('add_offset', None)
                        slice_data.encoding.pop('dtype', None)

                    print(f"  -> fetching data for {filename}")
                    slice_data.rio.to_raster(
                        filepath,
                        driver="GTiff",
                        compress="deflate",
                        tiled=True,
                        BIGTIFF="IF_SAFER",
                        lock=True
                    )
                    del slice_data
                    gc.collect()

                processed_files_count += 1
                append_to_log(filename, "success")
                month_save_end_time = time.time()
                print(f"    -> Saved {filename} (took {month_save_end_time - month_save_start_time:.2f}s)")

            except Exception as e:
                failed_files += 1
                failed_filenames.append(filename)
                append_to_log(filename, f"failed: {str(e)}")
                print(f"    -> ERROR saving {filename}: {e}")

        ds_year.close()

    except Exception as e:
        print(f"   ERROR processing year {year}: {e}")
        try:
            ds_year.close()
        except NameError:
            pass

    year_end_time = time.time()
    print(f"===== Year {year} finished (took {year_end_time - year_start_time:.2f}s) =====")

# --- Summary Table ---
total_end_time = time.time()
print("\n--- Summary Table ---")
print(f"{'Result':<15} | {'Count':>5}")
print("-" * 25)
print(f"{'Saved':<15} | {processed_files_count:>5}")
print(f"{'Skipped':<15} | {skipped_files:>5}")
print(f"{'Failed':<15} | {failed_files:>5}")
print("-" * 25)
print(f"Total runtime: {total_end_time - total_start_time:.2f} seconds.")

# --- Write summary to file ---
with open(SUMMARY_FILE, 'w') as f:
    f.write("--- Summary Table ---\n")
    f.write(f"{'Result':<15} | {'Count':>5}\n")
    f.write("-" * 25 + "\n")
    f.write(f"{'Saved':<15} | {processed_files_count:>5}\n")
    f.write(f"{'Skipped':<15} | {skipped_files:>5}\n")
    f.write(f"{'Failed':<15} | {failed_files:>5}\n")
    f.write("-" * 25 + "\n")
    f.write(f"Total runtime: {total_end_time - total_start_time:.2f} seconds\n\n")
    if failed_filenames:
        f.write("Failed files:\n")
        for fn in failed_filenames:
            f.write(f"  - {fn}\n")

